In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH
from configs.cellina_config import MODEL_ARGS as CELLINA_MODEL_ARGS, TRAIN_ARGS as CELLINA_TRAIN_ARGS, PLAN_KWARGS as CELLINA_PLAN_KWARGS
from configs.cpa_config import MODEL_ARGS as CPA_MODEL_ARGS, TRAIN_ARGS as CPA_TRAIN_ARGS, PLAN_KWARGS as CPA_PLAN_KWARGS

In [ ]:
import scanpy as sc

adata = sc.read_h5ad('/data2/a330d/datasets/crc/processed/crc_cosmx_wt.h5ad')
adata = adata[adata.obs.sid.isin([231, 232, 242])].copy()
adata.write_h5ad('/data2/a330d/datasets/crc/processed/crc_patient_loo.h5ad')

In [4]:
import cellina

In [4]:
set_seed(0)

In [5]:
DATASET_NAME = "crc"  # or "merfish"
MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained/loo_patients")

In [6]:
CRC_PATHS = [
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_221.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
    #os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_120.h5ad"),
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [7]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 2048
library_size = 'latent'
n_deg = 50

In [8]:
adata = sc.read_h5ad('/data2/a330d/datasets/crc/processed/crc_cosmx_wt.h5ad')

In [9]:
adata = adata[adata.obs[batch_key]!=110].copy()

In [10]:
import gc
gc.collect()

2313

In [11]:
adata

AnnData object with n_obs × n_vars = 2360478 × 2000
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68_CK8_18', 'Max.CD68_CK8_18', 'Mean.CD298_B2M', 'Max.CD298_B2M', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'version', 'dualfiles', 'Run_name', 'Run_Tissue_name', 'ISH.concentration', 'Dash', 'tissue', 'Panel', 'assay_type', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'falsecode_quantil

In [24]:
# 120, 210, 242
HOLDOUT_SIDS = [210, 242]

In [25]:
def split_indices(
    adata,
    holdout_slide,
    batch_key,
    seed=0,
):
    if holdout_slide not in adata.obs[batch_key].unique():
        raise ValueError(f"holdout_slide '{holdout_slide}' not found in adata.obs['{batch_key}'] values")

    is_holdout_slide = adata.obs[batch_key] == holdout_slide
    test_mask = is_holdout_slide

    all_idx = np.arange(adata.n_obs)
    test_idx = np.where(test_mask.values)[0]
    trainval_idx = np.setdiff1d(all_idx, test_idx)

    rng = np.random.default_rng(seed)
    n_trainval = trainval_idx.shape[0]
    n_val = max(1, int(0.1 * n_trainval))
    val_idx_rel = rng.choice(np.arange(n_trainval), size=n_val, replace=False)
    val_idx = trainval_idx[val_idx_rel]
    train_idx = np.setdiff1d(trainval_idx, val_idx)

    # annotate is_holdout in adata.obs
    adata.obs['is_holdout'] = False
    if len(test_idx) > 0:
        adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

    return train_idx, val_idx, test_idx

In [ ]:
model_names = ['cpa'] # ['cellina', 'cpa']
results = []
for slide_id in HOLDOUT_SIDS:
    for model_name in model_names:
        if model_name == 'cellina':
            model_class = 'cellina'
        elif model_name == 'cpa':
            model_class = 'cpa'
            adata.obs[batch_key] = adata.obs[batch_key].astype(str) # CPA requires batch_key to be string
            slide_id = str(slide_id) # CPA requires batch_key to be string
        else:
            model_class = 'cellina_graph'

        # 50 times * in print
        print(f"{'='*50} Holout slide: {slide_id} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_slide=slide_id,
                                                    batch_key=batch_key,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        save_dir = os.path.join(MODEL_ROOT, str(slide_id), model_name)

        # Train model
        if model_class == 'cellina':
            from cellina import Cellina
            model_args = CELLINA_MODEL_ARGS.copy()
            train_args = CELLINA_TRAIN_ARGS.copy()
            plan_kwargs = CELLINA_PLAN_KWARGS.copy()
            
            Cellina.setup_anndata(adata, 
                                    batch_key=batch_key, 
                                    labels_key=labels_key, 
                                    domains_key=domains_key, 
                                    spatial_obsm_key='spatial_x', 
                                    layer='counts')
            model = Cellina(adata, **model_args)

            # Add split info
            train_args['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            if plan_kwargs is not None:
                model.train(**train_args, plan_kwargs=plan_kwargs)
            else:
                model.train(**train_args)
            model.save(save_dir, overwrite=True)
        
        if model_class == 'cpa':
            import cpa
            model_args = CPA_MODEL_ARGS.copy()
            train_args = CPA_TRAIN_ARGS.copy()
            plan_kwargs = CPA_PLAN_KWARGS.copy()

            adata.obs['dose'] = 1.0 # NOTE: dummy dose for compatibility with CPA model
            adata.obs['data_split'] = 'train'
            
            adata.obs.iloc[splits[1], adata.obs.columns.get_loc('data_split')] = 'valid'
            adata.obs.iloc[splits[2], adata.obs.columns.get_loc('data_split')] = 'test'
            cpa.CPA.setup_anndata(adata,
                      perturbation_key=domains_key,
                      control_group='REF',
                      dosage_key='dose',
                      batch_key=batch_key,
                      categorical_covariate_keys=[labels_key],
                      is_count_data=True,
                      max_comb_len=1,
                     )
            model = cpa.CPA(adata,
                            split_key='data_split',
                            train_split='train',
                            valid_split='valid',
                            test_split='test',
                            **model_args)
            model.train(**train_args, plan_kwargs=plan_kwargs, save_path=save_dir)

================================================== Holout slide: 210 ==================================================


100%|██████████| 2360478/2360478 [00:01<00:00, 1805400.27it/s]


INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


2026-07-24 20:48:08 | [INFO] Global seed set to 6977
100%|██████████| 3/3 [00:01<00:00,  3.00it/s]
INFO: GPU available: True (cuda), used: True
2026-07-24 20:48:09 | [INFO] GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
2026-07-24 20:48:09 | [INFO] TPU available: False, using: 0 TPU cores
INFO: IPU available: False, using: 0 IPUs
2026-07-24 20:48:09 | [INFO] IPU available: False, using: 0 IPUs
INFO: HPU available: False, using: 0 HPUs
2026-07-24 20:48:09 | [INFO] HPU available: False, using: 0 HPUs
INFO: You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
2026-07-24 20:48:09 | [INFO] You are using a CUDA device ('NVIDIA GeForce RTX 4090') th

Epoch 10/100:   9%|▉         | 9/100 [13:56<2:21:18, 93.17s/it, v_num=1, recon=449, r2_mean=0.472, adv_loss=1.42, acc_pert=0.778, acc_coarse_type=0.882, acc_sid=0.784]

INFO: 
Epoch 00009: cpa_metric reached. Module best state updated.
2026-07-24 21:03:56 | [INFO] 
Epoch 00009: cpa_metric reached. Module best state updated.



disnt_basal = 1.3177919531638642
disnt_after = 2.4234319442209267
val_r2_mean = 0.47780313944533437
val_r2_var = 0.3917380177367837
Epoch 20/100:  19%|█▉        | 19/100 [29:44<2:05:46, 93.16s/it, v_num=1, recon=443, r2_mean=0.49, adv_loss=1.33, acc_pert=0.788, acc_coarse_type=0.886, acc_sid=0.803, val_recon=449, disnt_basal=1.32, disnt_after=2.42, val_r2_mean=0.478, val_KL=nan] 
disnt_basal = 1.3427524550418624
disnt_after = 2.41681426167932
val_r2_mean = 0.49006815726876835
val_r2_var = 0.41576880532658256
Epoch 30/100:  29%|██▉       | 29/100 [45:36<1:51:15, 94.02s/it, v_num=1, recon=440, r2_mean=0.498, adv_loss=1.33, acc_pert=0.79, acc_coarse_type=0.887, acc_sid=0.801, val_recon=444, disnt_basal=1.34, disnt_after=2.42, val_r2_mean=0.49, val_KL=nan] 
disnt_basal = 1.3373328999430003
disnt_after = 2.4130061500114857
val_r2_mean = 0.5011891780879035
val_r2_var = 0.42447034105211123
Epoch 40/100:  39%|███▉      | 39/100 [1:03:01<1:52:19, 110.48s/it, v_num=1, recon=442, r2_mean=0.488, 

INFO: 
Epoch 00039: cpa_metric reached. Module best state updated.
2026-07-24 21:53:11 | [INFO] 
Epoch 00039: cpa_metric reached. Module best state updated.



disnt_basal = 0.8654199944550816
disnt_after = 2.397337387418874
val_r2_mean = 0.4806130313987935
val_r2_var = 0.41446210570308173
Epoch 50/100:  49%|████▉     | 49/100 [1:19:35<1:22:30, 97.07s/it, v_num=1, recon=440, r2_mean=0.494, adv_loss=3.5, acc_pert=0.425, acc_coarse_type=0.618, acc_sid=0.437, val_recon=443, disnt_basal=0.865, disnt_after=2.4, val_r2_mean=0.481, val_KL=nan]  
disnt_basal = 0.8554757028344695
disnt_after = 2.393768749109341
val_r2_mean = 0.4117405868585244
val_r2_var = 0.3032210873068103
Epoch 54/100:  53%|█████▎    | 53/100 [1:31:50<2:08:35, 164.17s/it, v_num=1, recon=440, r2_mean=0.493, adv_loss=3.55, acc_pert=0.406, acc_coarse_type=0.615, acc_sid=0.43, val_recon=448, disnt_basal=0.855, disnt_after=2.39, val_r2_mean=0.412, val_KL=nan] 

In [ ]:
model_names = ['cellina']
results = []
for slide_id in HOLDOUT_SIDS:
    for model_name in model_names:
        model_class = 'cellina' if model_name == 'cellina' else 'cellina_graph'
        # 50 times * in print
        print(f"{'='*50} Holout slide: {slide_id} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_slide=slide_id,
                                                    batch_key=batch_key,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        save_dir = os.path.join(MODEL_ROOT, str(slide_id), model_name)
        # Load model and eval
        try:
            model = _load_model(save_dir,
                                model_class=model_class,
                                adata=adata,
                                splits=splits)
        except Exception as e:
            print(f"Failed to load model from {save_dir} with error: {e}")
            continue
        for holdout_celltype in HOLDOUT_CELLTYPES:
            print(f"{'='*50} Holdout celltype: {holdout_celltype}{'='*50}")
            adata_holdout = adata[adata.obs[batch_key] == slide_id]
            is_control_region = adata_holdout.obs[domains_key]==(control_domain)
            is_holdout_ct = adata_holdout.obs[labels_key].astype(str) == holdout_celltype
            mask_control = is_control_region & is_holdout_ct
            idx_control = np.where(mask_control.values)[0]    
            
            for hd in holdout_domains:
                is_holdout_region = adata_holdout.obs[domains_key].astype(str) == hd
                mask_ct_target = is_holdout_ct & is_holdout_region
                idx_target = np.where(mask_ct_target.values)[0]

                # "neighbour_indices" are indices of the neighbors of idx_target cells
                step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
                preprocess_spatial_features(adata_holdout, step_size_px=step_size_px, n_neighbors=200, test_indices=idx_target)
                conn = adata_holdout.obsp["spatial_connectivities_orig"]
                sub_conn = conn[idx_target]                # rows for target cells
                neighbor_indices = sub_conn.nonzero()[1]   # all neighbors at once
                neighbor_indices = np.unique(neighbor_indices)
                # keep only non-holdout-ct neighbors
                neighbor_indices = neighbor_indices[~is_holdout_ct.values[neighbor_indices]]

                args_gex = {
                    "adata": adata_holdout,
                    "indices": idx_control,
                    "batch_size": batch_size,
                    "seed": 0,
                    "neighbour_indices": neighbor_indices
                }
                if model_class.lower() == 'cellina_graph':
                    args_gex["n_neighbors_per_seed"] = 50
                else:
                    args_gex['precomputed'] = True
                
                cf_counts = model.get_counterfactual_expression(**args_gex)
                
                # Compute stats
                control = adata_holdout.layers['counts'][mask_control.values, :]
                target = adata_holdout.layers['counts'][mask_ct_target.values, :]
                control, target = _to_dense(control), _to_dense(target)
                counterfactual = cf_counts

                gt_lfc, cf_lfc, deg = get_lfc(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)

                spear, _ = spearmanr(gt_lfc[deg], cf_lfc[deg])
                pear, _ = pearsonr(gt_lfc[deg], cf_lfc[deg])
                prec = precision(gt_lfc, cf_lfc, k=n_deg, use_abs=True)
                dir_match = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="intersection")
                dir_match_k = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="k")
                dir_match_gt = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="gt_topk")
                mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=COUNTS_PER_K)
                edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K)
                edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True)
                edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True)
                edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True, log1p=False)
                rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg, library_size=COUNTS_PER_K)
                mse_lfc = compute_mse_lfc(gt_vec=gt_lfc, cf_vec=cf_lfc, deg=deg)

                results.append(
                        dict(
                        dataset_name=DATASET_NAME,
                        sid=slide_id,
                        control_domain=control_domain,
                        target_domain=hd,
                        n_deg=n_deg,
                        model_name=model_name,
                        holdout_celltype=holdout_celltype,
                        spearman=spear,
                        pearson=pear,
                        precision=prec,
                        direction_match=dir_match,
                        direction_match_k=dir_match_k,
                        direction_match_gt=dir_match_gt,
                        mixing_index=mix_idx,
                        edistance_global=edist_global,
                        edistance_local=edist_local,
                        edistance_pca_log=edist_pca_log,
                        edistance_pca=edist_pca,
                        rmse=rmse,
                        mse_lfc=mse_lfc,
                        )
                )
            gc.collect()

INFO: Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
2026-07-24 15:39:29 | [INFO] Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


================================================== Holout slide: 210 ==================================================
INFO     File /data2/a330d/data/ood/trained/loo_patients/210/cellina/model.pt already downloaded                   
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data2/a330d/data/ood/trained/loo_patients/210/cellina
================================================== Holdout celltype: Endothelial==================================================


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
================================================== Holdout celltype: Epithelial==================================================


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


...

INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
================================================== Holdout celltype: Myeloid==================================================


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
================================================== Holdout celltype: T_cell==================================================


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


In [17]:
# Append to existing csv if exists, otherwise create new csv
results_csv_name = f'../../results/loo_cellina_{DATASET_NAME}_DEG_{n_deg}_patients.csv'

df_results = pd.DataFrame(results)
if os.path.exists(results_csv_name):
    df_results.to_csv(f"{results_csv_name}", index=False, mode='a', header=False)
else:
    df_results.to_csv(f"{results_csv_name}", index=False)

In [18]:
df_results

,dataset_name,sid,control_domain,target_domain,n_deg,model_name,holdout_celltype,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse,mse_lfc
0,crc,210,REF,CRC,50,cellina,Endothelial,0.907899,0.994052,0.58,1.000000,0.58,1.00,0.983414,74.929626,78.344838,7.105014,163.485356,5980.534829,0.979571
1,crc,210,REF,CRC,50,cellina,Epithelial,0.717071,0.783099,0.40,1.000000,0.40,0.96,0.779665,60.812232,60.593043,6.903071,420.299672,225555.244069,6.069775
2,crc,210,REF,CRC,50,cellina,Fibroblast,0.604130,0.875093,0.26,1.000000,0.26,0.98,0.713381,66.618017,66.250997,7.243655,388.409317,33438.700647,1.329255
3,crc,210,REF,CRC,50,cellina,Myeloid,0.609508,0.875953,0.36,1.000000,0.36,0.86,0.838621,70.088205,71.846670,8.039888,265.014933,20985.027183,3.686190
4,crc,210,REF,CRC,50,cellina,T_cell,0.713517,0.968019,0.30,0.933333,0.28,0.94,0.809435,77.220521,81.524417,7.603711,156.916536,7992.266771,1.436809
5,crc,242,REF,CRC,50,cellina,Endothelial,0.694406,0.865467,0.22,1.000000,0.22,0.96,0.432151,71.163434,72.918226,5.166230,259.205915,3289.496768,2.722420
6,crc,242,REF,CRC,50,cellina,Epithelial,0.784586,0.856111,0.30,1.000000,0.30,0.96,0.829225,62.754711,64.312808,7.856952,322.069641,78207.445271,8.077026
7,crc,242,REF,CRC,50,cellina,Fibroblast,0.786699,0.766485,0.26,1.000000,0.26,0.96,0.582594,68.942240,67.688794,5.479370,540.738716,35050.977977,2.388160
8,crc,242,REF,CRC,50,cellina,Myeloid,0.781032,0.813057,0.22,1.000000,0.22,0.98,0.644120,64.917211,66.420374,4.624782,292.175106,5612.273140,2.071337
9,crc,242,REF,CRC,50,cellina,T_cell,0.765858,0.751222,0.18,1.000000,0.18,0.82,0.681028,80.336273,86.126943,4.015850,143.244313,19128.079401,1.202406


# Plot table

In [10]:
import os
import pandas as pd
import numpy as np

In [11]:
# Define the directory containing the CSV files
csv_dir = '../../results/loo_patient'

# List all CSV files in the directory
csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# Read and concatenate all CSV files into a single DataFrame
df_list = []
for csv_file in csv_files:
    df = pd.read_csv(os.path.join(csv_dir, csv_file))
    df_list.append(df)
combined_df = pd.concat(df_list, ignore_index=True)


In [12]:
combined_df['rmse_lfc'] = np.sqrt(combined_df['mse_lfc'])
combined_df = combined_df.rename(columns={"direction_match_k": "signed_precision"})
combined_df = combined_df.rename(columns={"edistance_pca_log": "e-distance"})

In [13]:
metrics = ['pearson', 'signed_precision', 'e-distance', 'rmse_lfc']

In [14]:
summary = combined_df.groupby("model_name")[metrics].agg(["mean", "std"])

# Format as "mean ± std" strings
table = pd.DataFrame(index=summary.index)
for m in metrics:
    table[m] = summary[(m, "mean")].round(2).astype(str) + " ± " + summary[(m, "std")].round(2).astype(str)

print(table)

                pearson signed_precision     e-distance     rmse_lfc
model_name                                                          
baseline    0.37 ± 0.32       0.1 ± 0.08  31.83 ± 11.43  6.16 ± 3.55
cellina     0.61 ± 0.19      0.24 ± 0.11    7.23 ± 1.88   1.8 ± 0.74
cpa         0.48 ± 0.19      0.06 ± 0.05   15.55 ± 4.63  2.94 ± 0.59
